# 01- Experiment Objective

# 02 Experimental Definition

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.core.utility_experiment_config import UtilityExperimentConfig
from src.core.dataset_config import DatasetConfig
from src.core.preprocessing_config import PreprocessingConfig
from src.core.task_config import TaskConfig


In [3]:
CATEGORICAL_COLUMNS_FOR_CLASSIFICATION = [
    "TP_COR_RACA",
    "TP_NACIONALIDADE",
    "TP_ESTADO_CIVIL",
    "TP_ST_CONCLUSAO",
    "TP_ENSINO",
    "IN_TREINEIRO",
    "Q001",
    "Q002",
    "Q003",
    "Q004",
    "Q005",
    "Q006",
    "Q007",
    "Q008",
    "Q009",
    "Q010",
    "Q011",
    "Q012",
    "Q013",
    "Q014",
    "Q015",
    "Q016",
    "Q017",
    "Q018",
    "Q019",
    "Q020",
    "Q021",
    "Q022",
    "Q023",
]

NUMERICAL_COLUMNS_FOR_CLASSIFICATION = [
    "TP_FAIXA_ETARIA",
    "TP_ANO_CONCLUIU",
]

In [4]:
CATEGORICAL_COLUMNS_FOR_REGRESSION = [
    "TP_SEXO",
    "TP_COR_RACA",
    "TP_NACIONALIDADE",
    "TP_ESTADO_CIVIL",
    "TP_ST_CONCLUSAO",
    "TP_ENSINO",
    "IN_TREINEIRO",
    "Q001",
    "Q002",
    "Q003",
    "Q004",
    "Q006",
    "Q007",
    "Q008",
    "Q009",
    "Q010",
    "Q011",
    "Q012",
    "Q013",
    "Q014",
    "Q015",
    "Q016",
    "Q017",
    "Q018",
    "Q019",
    "Q020",
    "Q021",
    "Q022",
    "Q023",
]

NUMERICAL_COLUMNS_FOR_REGRESSION = [
    "TP_FAIXA_ETARIA",
    "TP_ANO_CONCLUIU",
]

# Experiment Config

In [5]:
def get_regression_config():

    dataset_config = DatasetConfig(
        dataset_name="enem",
        dataset_version="enem_2025 - v-2026-07-21_23-19-59",
        data_sample_size=100_000,
        data_random_state=42,
    )
    
    tasks_config = [
        TaskConfig(task_type="regression", target="Q005"),
    ]

    preprocessing_config = PreprocessingConfig(
        categorical_columns=CATEGORICAL_COLUMNS_FOR_REGRESSION,
        numerical_columns=NUMERICAL_COLUMNS_FOR_REGRESSION,
    )

    return UtilityExperimentConfig(
        dataset=dataset_config,
        tasks=tasks_config,
        preprocessing=preprocessing_config,
    )

In [6]:
def get_classification_config():

    dataset_config = DatasetConfig(
        dataset_name="enem",
        dataset_version="enem_2025 - v-2026-07-21_23-19-59",
        data_sample_size=100_000,
        data_random_state=42,
    )
    
    tasks_config = [
        TaskConfig(task_type="classification", target="TP_SEXO"),
    ]

    preprocessing_config = PreprocessingConfig(
        categorical_columns=CATEGORICAL_COLUMNS_FOR_CLASSIFICATION,
        numerical_columns=NUMERICAL_COLUMNS_FOR_CLASSIFICATION,
    )

    return UtilityExperimentConfig(
        dataset=dataset_config,
        tasks=tasks_config,
        preprocessing=preprocessing_config,
    )

# Select experiment

In [7]:
evaluation_config = get_classification_config()

03 Dataset Preparation

In [8]:
from src.data.dataset_registry import load_dataset_bundle
from src.data.sample_dataset import sample_dataset_bundle

In [9]:
dataset_bundle = load_dataset_bundle(
    dataset_name=evaluation_config.dataset.dataset_name,
    dataset_version=evaluation_config.dataset.dataset_version,
)

dataset_sample = sample_dataset_bundle(
    dataset_bundle,
    sample_size=evaluation_config.dataset.data_sample_size,
    random_state=evaluation_config.dataset.data_random_state,
)

dataset_sample

{'dataset_path': PosixPath('/home/LR/Projects/IC Privacidade/Repositories/ML-MIA-Privacy-Evaluation/src/data/datasets/enem/enem_2025 - v-2026-07-21_23-19-59'),
 'dataset_version': 'enem_2025 - v-2026-07-21_23-19-59',
 'datasets': [      TP_SEXO  TP_COR_RACA  TP_NACIONALIDADE SG_UF_PROVA Q023  \
  0           M            1                 1          SC    E   
  1           F            1                 1          PI    A   
  2           M            2                 1          CE    A   
  3           M            1                 1          SP    A   
  4           F            3                 1          RJ    D   
  ...       ...          ...               ...         ...  ...   
  99995       F            3                 1          SP    D   
  99996       F            1                 1          RS    D   
  99997       F            1                 1          MG    A   
  99998       F            3                 1          MA    A   
  99999       F            1      

04 Feature Preparation

In [10]:
from src.core.splits_config import SplitConfig

split_plan = SplitConfig(seed=42, test_size=0.5)

In [11]:
from src.experiments.utility_evaluation_services import feature_preparation

prepared_features = []

for dataset_name, df in zip(dataset_bundle['dataset_names'], dataset_bundle['datasets']):
    for task in evaluation_config.tasks:
        prepared = feature_preparation.prepare_features(
            name=dataset_name,
            df=df,
            task_config=task,
            split_plan=split_plan,
            preprocessing_config=evaluation_config.preprocessing,
        )
        prepared_features.append(prepared)


print(f"Prerpared features: {len(prepared_features)}")

/home/LR/Projects/IC Privacidade/Repositories/ML-MIA-Privacy-Evaluation/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/LR/Projects/IC Privacidade/Repositories/ML-MIA-Privacy-Evaluation/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/LR/Projects/IC Privacidade/Repositories/ML-MIA-Privacy-Evaluation/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/LR/Projects/IC Privacidade/Repositories/ML-MIA-Privacy-Evaluation/.venv/l

Prerpared features: 5


/home/LR/Projects/IC Privacidade/Repositories/ML-MIA-Privacy-Evaluation/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


05 Model execution

In [12]:
from src.core.models_spec_config import ModelSpec

# Classification Models

In [13]:
XGBOOST_CLASSIFIER = ModelSpec(
    name="xgboost",
    model_type="xgboost_classifier",
    parameters={
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "tree_method": "hist",
        "n_jobs": 4,
        "verbosity": 0,
    },
)


# Regression Models

In [14]:
XGBOOST_REGRESSOR = ModelSpec(
    name="xgboost",
    model_type="xgboost_regressor",
    parameters={
        "objective": "reg:squarederror",
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "tree_method": "hist",
        "n_jobs": 4,
        "verbosity": 0,
    },
)

In [15]:
CLASSIFICATION_MODELS = [
    XGBOOST_CLASSIFIER,
]


REGRESSION_MODELS = [
    XGBOOST_REGRESSOR,
]

# Running models

In [16]:
from src.experiments.utility_evaluation_services.model import model_runner

06 Utility Evaluation

In [17]:
from src.experiments.utility_evaluation_services import metrics

from src.core.results_config import UtilityClassificationResult, UtilityRegressionResult


In [18]:
utility_records = []
leakage_input= []

for prepared in prepared_features:

    models = (
        CLASSIFICATION_MODELS
        if prepared.task_type == "classification"
        else REGRESSION_MODELS
    )

    for model_spec in models:

        prediction = model_runner.execute_model(
            prepared_features=prepared,
            model_spec=model_spec,
        )

        utility = metrics.compute_utility_metrics(
            prediction_result=prediction,
            task_type=prepared.task_type,
        )

        record  = {
            "dataset": prepared.name,
            "task_type": prepared.task_type,
            "target": prepared.target,
            "model": model_spec.name,
            "model_type": model_spec.model_type,
        }

        if type(utility)  == UtilityClassificationResult:
            record.update({
                "test_acc": utility.test_acc,
                "train_acc": utility.train_acc,
                "validation_acc": utility.validation_acc,
                "test_precision": utility.test_precision,
                "test_recall": utility.test_recall,
                "test_f1": utility.test_f1,
                "generalization_gap_%": utility.generalization_gap,
            })


            leakage_input.append({
                "dataset": prepared.name,
                "task_type": prepared.task_type,
                "target": prepared.target,
                "model": model_spec.name,
                "model_type": model_spec.model_type,

                "X_pool": prepared.X_train,
                "y_pool": prepared.y_train,

                "target_prediction": {
                    "train_proba": prediction.train_proba,
                    "test_proba": prediction.test_proba,
                    "y_train_encoded": prediction.y_train_encoded,
                    "y_test_encoded": prediction.y_test_encoded,
                },
    })

            
        elif type(utility) == UtilityRegressionResult:
            record.update({
                "test_mae": utility.test_mae,
                "test_r2_score": utility.test_r2,
                "train_r2_score": utility.train_r2,
                "validation_r2_score": utility.validation_r2,
                "generalization_gap_%": utility.generalization_gap,
            })

            leakage_input.append({
                "dataset": prepared.name,
                "task_type": prepared.task_type,
                "target": prepared.target,
                "model": model_spec.name,
                "model_type": model_spec.model_type,

                "X_pool": prepared.X_train,
                "y_pool": prepared.y_train,

                "target_prediction": {
                    "y_train_true": prediction.y_train_true,
                    "y_train_pred": prediction.y_train_pred,
                    "y_test_true": prediction.y_test_true,
                    "y_test_pred": prediction.y_test_pred,
                },
            })

        utility_records.append(record)


utility_records

[{'dataset': 'baseline',
  'task_type': 'classification',
  'target': 'TP_SEXO',
  'model': 'xgboost',
  'model_type': 'xgboost_classifier',
  'test_acc': 0.62132,
  'train_acc': 0.65274,
  'validation_acc': 0.62212,
  'test_precision': 0.6042952238395747,
  'test_recall': 0.62132,
  'test_f1': 0.5350458071181334,
  'generalization_gap_%': 3.1420000000000003},
 {'dataset': 'dp_eps_0.1',
  'task_type': 'classification',
  'target': 'TP_SEXO',
  'model': 'xgboost',
  'model_type': 'xgboost_classifier',
  'test_acc': 0.60404,
  'train_acc': 0.62716,
  'validation_acc': 0.60452,
  'test_precision': 0.5744009751966905,
  'test_recall': 0.60404,
  'test_f1': 0.4678567992637216,
  'generalization_gap_%': 2.312000000000003},
 {'dataset': 'dp_eps_0.5',
  'task_type': 'classification',
  'target': 'TP_SEXO',
  'model': 'xgboost',
  'model_type': 'xgboost_classifier',
  'test_acc': 0.60308,
  'train_acc': 0.62996,
  'validation_acc': 0.603,
  'test_precision': 0.5721370343796636,
  'test_recall':

07 Visualization

08 Export

In [19]:
import pandas as pd
from datetime import datetime

from artifacts.persistence import persist_utility_artifact

EXPERIMENT_TYPE = "utility_evaluation"
ARTIFACT_SCHEMA_VERSION = "1.0"

experiment_id =  datetime.now().strftime("%Y%m%d_%H%M%S")

utility_metrics = pd.DataFrame(utility_records)
utility_metrics.insert(0, "experiment_id", experiment_id)

input_leakage= pd.DataFrame(leakage_input)

model_specs = {
    (model.name, model.model_type): {
        "name": model.name,
        "model_type": model.model_type,
        "parameters": model.parameters,
    }
    for model in [*CLASSIFICATION_MODELS, *REGRESSION_MODELS]
}

experiment_metadata = {
    "experiment_id": experiment_id,
    "experiment_type": EXPERIMENT_TYPE,
    "artifact_schema_version": ARTIFACT_SCHEMA_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset": {
        "name": evaluation_config.dataset.dataset_name,
        "version": evaluation_config.dataset.dataset_version,
        "sample_size": evaluation_config.dataset.data_sample_size,
        "random_state": evaluation_config.dataset.data_random_state,
    },
    "split": {
        "seed": split_plan.seed,
        "test_size": split_plan.test_size,
    },
    "preprocessing": {
        "categorical_columns": evaluation_config.preprocessing.categorical_columns,
        "numerical_columns": evaluation_config.preprocessing.numerical_columns,
    },
    "tasks": [
        {"task_type": task.task_type, "target": task.target}
        for task in evaluation_config.tasks
    ],
    "models": list(model_specs.values()),
}

artifact_path = persist_utility_artifact(
    experiment_id=experiment_id,
    metadata=experiment_metadata,
    utility_metrics=utility_metrics,
    input_leakage= input_leakage
)

artifact_path

PosixPath('/home/LR/Projects/IC Privacidade/Repositories/ML-MIA-Privacy-Evaluation/artifacts/utility/utility_20260827_002700')

10 Conclusion / Observations